# SAP ECC Data Loader

Dit notebook:
1. Leest de SAP data dictionary (DD03L, sap_tables.md) voor veld-definities, primary keys en foreign keys
2. Genereert DDL (CREATE OR REPLACE TABLE) in schema SAP_ECC
3. Laadt de CSV bestanden uit sap/data/ in de aangemaakte tabellen

In [ ]:
import csv
import os
import re
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Stel database in
session.sql("CREATE DATABASE IF NOT EXISTS SAP").collect()
session.sql("USE DATABASE SAP").collect()
session.sql("USE SCHEMA PUBLIC").collect()

# Workspace bestandssysteem pad
WORKSPACE_ID = os.environ.get('SNOWFLAKE_WORKSPACE_ID', '')
FS_ROOT = f'/filesystem/{WORKSPACE_ID}'
DD_PATH = f'{FS_ROOT}/sap/dd'
DATA_PATH = f'{FS_ROOT}/sap/data'
SCHEMA = 'SAP.SAP_ECC'

print(f'Database: SAP')
print(f'Schema: {SCHEMA}')
print(f'DD pad: {DD_PATH}')
print(f'Data pad: {DATA_PATH}')
print(f'DD bestanden: {os.listdir(DD_PATH)}')
print(f'Data bestanden (eerste 10): {sorted(os.listdir(DATA_PATH))[:10]}')

In [ ]:
# Lees DD03L - velddefinities per tabel (direct van filesystem)
dd03l = pd.read_csv(f'{DD_PATH}/DD03L.csv')
print(f'DD03L: {len(dd03l)} veldefinities geladen')
print(f'Tabellen: {dd03l["TABNAME"].nunique()}')
dd03l.head(10)

In [ ]:
# Parse sap_tables.md voor primary keys en foreign keys
with open(f'{DD_PATH}/sap_tables.md', 'r') as f:
    lines = f.readlines()

# Parse markdown tabel
pk_lookup = {}
fk_lookup = {}

for line in lines:
    line = line.strip()
    if not line.startswith('|'):
        continue
    parts = [p.strip() for p in line.split('|')]
    # parts[0] is leeg (voor eerste |), parts[-1] is leeg (na laatste |)
    parts = [p for p in parts if p]  # verwijder lege strings
    if len(parts) < 4:
        continue
    sap_table = parts[0]
    # Skip header en separator rijen
    if sap_table in ('SAP Table', '---', '') or sap_table.startswith('---'):
        continue
    if not re.match(r'^[A-Z0-9_]+$', sap_table):
        continue
    
    pk_str = parts[2] if len(parts) > 2 else ''
    fk_str = parts[3] if len(parts) > 3 else ''
    pk_lookup[sap_table] = pk_str
    fk_lookup[sap_table] = fk_str

print(f'sap_tables.md: {len(pk_lookup)} tabellen met PK/FK info')
print('\nVoorbeeld:')
for t in list(pk_lookup.keys())[:5]:
    print(f'  {t}: PK={pk_lookup[t]}, FK={fk_lookup[t]}')

In [ ]:
# SAP datatype naar Snowflake datatype mapping
def sap_to_snowflake_type(datatype, leng, decimals):
    mapping = {
        'CHAR': f'VARCHAR({leng})',
        'NUMC': f'VARCHAR({leng})',  # Numeriek karakter, bewaar als string
        'CLNT': f'VARCHAR({leng})',
        'DATS': 'VARCHAR(8)',        # SAP datum YYYYMMDD
        'TIMS': 'VARCHAR(6)',        # SAP tijd HHMMSS
        'DEC': f'NUMBER({leng},{decimals})',
        'QUAN': f'NUMBER({leng},{decimals})',
        'CURR': f'NUMBER({leng},{decimals})',
        'INT1': 'NUMBER(3,0)',
        'INT2': 'NUMBER(5,0)',
        'INT4': 'NUMBER(10,0)',
        'INT8': 'NUMBER(19,0)',
        'FLTP': 'FLOAT',
        'UNIT': f'VARCHAR({leng})',
        'CUKY': f'VARCHAR({leng})',
        'LANG': 'VARCHAR(1)',
        'STRG': 'VARCHAR(16777216)',
        'SSTR': f'VARCHAR({max(leng, 256)})',
        'RSTR': 'VARCHAR(16777216)',
        'LCHR': 'VARCHAR(16777216)',
        'LRAW': 'VARCHAR(16777216)',
        'RAW': f'VARCHAR({leng * 2})',
    }
    return mapping.get(datatype, f'VARCHAR({max(leng, 50)})') 

print('Datatype mapping functie aangemaakt')

In [ ]:
# Genereer DDL statements
def generate_ddl(table_name, fields_df, pk_str, fk_str):
    """Genereer CREATE OR REPLACE TABLE statement."""
    cols = []
    for _, row in fields_df.iterrows():
        sf_type = sap_to_snowflake_type(row['DATATYPE'], row['LENG'], row['DECIMALS'])
        not_null = ' NOT NULL' if row.get('NOTNULL') == 'X' else ''
        cols.append(f'    {row["FIELDNAME"]} {sf_type}{not_null}')
    
    # Primary key constraint
    constraints = []
    if pk_str and pk_str != 'None':
        pk_cols = ', '.join([c.strip() for c in pk_str.split(',')])
        constraints.append(f'    CONSTRAINT PK_{table_name} PRIMARY KEY ({pk_cols})')
    
    # Foreign key constraints
    if fk_str and fk_str != 'None' and fk_str != '':
        fk_parts = fk_str.split(';')
        for i, fk in enumerate(fk_parts):
            fk = fk.strip()
            if '->' not in fk:
                continue
            src_part, tgt_part = fk.split('->')
            src_cols = ', '.join([c.strip() for c in src_part.strip().split(',')])
            tgt_table = tgt_part.strip().split(',')[0].strip() if ',' not in tgt_part.strip() else tgt_part.strip().split(',')[0].strip()
            
            # Bepaal target kolommen (zelfde als source kolommen als ze naar de PK verwijzen)
            tgt_cols_raw = tgt_part.strip().split(',')
            if len(tgt_cols_raw) > 1:
                # FK verwijst naar meerdere kolommen in target: bv "BUKRS, BELNR, GJAHR -> BKPF"
                tgt_table = tgt_cols_raw[-1].strip()
                tgt_cols = src_cols  # FK cols = source cols
            else:
                tgt_table = tgt_part.strip()
                tgt_cols = src_cols
            
            constraints.append(
                f'    CONSTRAINT FK_{table_name}_{i+1} FOREIGN KEY ({src_cols}) REFERENCES {SCHEMA}.{tgt_table} ({tgt_cols})'
            )
    
    all_parts = cols + constraints
    ddl = f'CREATE OR REPLACE TABLE {SCHEMA}.{table_name} (\n'
    ddl += ',\n'.join(all_parts)
    ddl += '\n);'
    return ddl

# Genereer alle DDL statements
ddl_statements = []
tables_in_dd = dd03l['TABNAME'].unique()

for table_name in sorted(tables_in_dd):
    fields = dd03l[dd03l['TABNAME'] == table_name].sort_values('POSITION')
    pk_str = pk_lookup.get(table_name, '')
    fk_str = fk_lookup.get(table_name, '')
    ddl = generate_ddl(table_name, fields, pk_str, fk_str)
    ddl_statements.append((table_name, ddl))

print(f'{len(ddl_statements)} DDL statements gegenereerd')
print('\n--- Voorbeeld (AFKO) ---')
print(ddl_statements[0][1])

In [ ]:
# Maak schema aan en voer alle DDL uit
session.sql(f'CREATE SCHEMA IF NOT EXISTS {SCHEMA}').collect()
print(f'Schema {SCHEMA} aangemaakt/bevestigd')

# Voer DDL uit in juiste volgorde (tabellen zonder FK eerst)
# Sorteer: tabellen zonder FK relaties eerst
tables_no_fk = [(t, ddl) for t, ddl in ddl_statements if fk_lookup.get(t, '') in ('None', '', None)]
tables_with_fk = [(t, ddl) for t, ddl in ddl_statements if fk_lookup.get(t, '') not in ('None', '', None)]
ordered_ddl = tables_no_fk + tables_with_fk

success_count = 0
error_count = 0
errors = []

for table_name, ddl in ordered_ddl:
    try:
        session.sql(ddl).collect()
        success_count += 1
    except Exception as e:
        error_count += 1
        errors.append((table_name, str(e)))

print(f'\nResultaat: {success_count} tabellen aangemaakt, {error_count} fouten')
if errors:
    print('\nFouten:')
    for t, e in errors[:10]:
        print(f'  {t}: {e}')

In [ ]:
# Laad CSV data in de tabellen
# Lees CSV van filesystem en schrijf via write_pandas naar Snowflake
from snowflake.snowpark.functions import col

load_success = 0
load_errors = []
load_skipped = 0

# Lijst van CSV bestanden in sap/data/
csv_files = [f for f in sorted(os.listdir(DATA_PATH)) if f.endswith('.csv')]
print(f'{len(csv_files)} CSV bestanden gevonden in {DATA_PATH}')

table_names_set = set(t for t, _ in ddl_statements)

for filename in csv_files:
    table_name = filename.replace('.csv', '').upper()
    
    if table_name not in table_names_set:
        load_skipped += 1
        continue
    
    try:
        df = pd.read_csv(f'{DATA_PATH}/{filename}', dtype=str, keep_default_na=False)
        df.columns = [c.upper() for c in df.columns]
        
        if len(df) == 0:
            load_skipped += 1
            continue
        
        # Schrijf naar Snowflake via write_pandas
        session.write_pandas(
            df, 
            table_name=table_name, 
            database='SAP',
            schema='SAP_ECC',
            overwrite=True,
            auto_create_table=False
        )
        load_success += 1
        print(f'  {table_name}: {len(df)} rijen geladen')
    except Exception as e:
        load_errors.append((table_name, str(e)))

print(f'\n--- Samenvatting ---')
print(f'Geladen: {load_success} tabellen')
print(f'Overgeslagen: {load_skipped}')
print(f'Fouten: {len(load_errors)}')
if load_errors:
    print('\nLaadfouten:')
    for t, e in load_errors[:10]:
        print(f'  {t}: {e}')

In [ ]:
# Verificatie: toon rij-aantallen per tabel
verify_results = []
for table_name, _ in ddl_statements:
    try:
        count = session.sql(f'SELECT COUNT(*) as CNT FROM {SCHEMA}.{table_name}').to_pandas()['CNT'][0]
        verify_results.append({'TABLE': table_name, 'ROW_COUNT': count})
    except:
        verify_results.append({'TABLE': table_name, 'ROW_COUNT': -1})

import pandas as pd
verify_df = pd.DataFrame(verify_results)
print(f'Totaal rijen geladen: {verify_df["ROW_COUNT"][verify_df["ROW_COUNT"] > 0].sum()}')
print(f'Tabellen met data: {len(verify_df[verify_df["ROW_COUNT"] > 0])}')
print(f'Lege tabellen: {len(verify_df[verify_df["ROW_COUNT"] == 0])}')
verify_df[verify_df['ROW_COUNT'] > 0].sort_values('ROW_COUNT', ascending=False).head(20)